# اليوم 2، المعمل 2B: التصنيف وفرط التعلّم

التصنيف يعيد احتمالًا أولًا. تحويل الاحتمال إلى 0 أو 1 يحتاج عتبة مرتبطة بالقرار.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "data" / "raw"
RANDOM_STATE = 42


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from manafeth.data import load_customers, split_customers
from manafeth.features import FEATURE_COLUMNS, build_preprocessor

df = load_customers(DATA)
X_train, X_test, y_train, y_test = split_customers(df)
X_fit, X_valid, y_fit, y_valid = __import__('sklearn').model_selection.train_test_split(X_train, y_train, test_size=.25, stratify=y_train, random_state=RANDOM_STATE)


## Logistic Regression

ابن Pipeline يجمع التجهيز والنموذج. احسب PR-AUC من الاحتمالات، وليس من 0/1.

In [ ]:
logit = Pipeline([("prep", build_preprocessor()), ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))])
logit.fit(X_fit, y_fit)
valid_prob = logit.predict_proba(X_valid)[:, 1]
print("Logistic validation PR-AUC:", round(average_precision_score(y_valid, valid_prob), 3))

train_scores, valid_scores = [], []
for depth in range(1, 21):
    tree = Pipeline([("prep", build_preprocessor(scale_numeric=False)), ("model", DecisionTreeClassifier(max_depth=depth, min_samples_leaf=20, random_state=RANDOM_STATE))])
    tree.fit(X_fit, y_fit)
    train_scores.append(average_precision_score(y_fit, tree.predict_proba(X_fit)[:, 1]))
    valid_scores.append(average_precision_score(y_valid, tree.predict_proba(X_valid)[:, 1]))
plt.plot(range(1, 21), train_scores, label="train")
plt.plot(range(1, 21), valid_scores, label="validation")
plt.xlabel("max_depth"); plt.ylabel("PR-AUC"); plt.legend(); plt.show()
print("best depth:", int(pd.Series(valid_scores, index=range(1, 21)).idxmax()))


## تجربة عمق الشجرة

غيّر `max_depth` من 1 إلى 20. سجّل train وvalidation PR-AUC. ابحث عن الفجوة التي تكبر بعد أن تتحسن نتيجة التدريب وحدها.

In [ ]:
# أُنجزت خطوات هذا القسم في خلية الحل السابقة.


**سؤال التسليم:** اختر عمقًا وادعمه بدليل من validation، لا من train.